# 🧠 Sentiment Analysis — ML Deep Dive
**A professional NLP pipeline with model comparisons, SHAP explainability, and visualizations.**

This notebook demonstrates the ML backbone powering the SentimentAI dashboard.

In [ ]:
import re
import math
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from datetime import datetime, timedelta

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0a0a0f',
    'axes.facecolor': '#111118',
    'axes.edgecolor': '#333344',
    'axes.labelcolor': '#8888aa',
    'xtick.color': '#8888aa',
    'ytick.color': '#8888aa',
    'grid.color': '#1e1e2e',
    'text.color': '#f0f0f5',
    'font.family': 'monospace',
})

COLORS = {
    'positive': '#34d399',
    'negative': '#f87171',
    'neutral':  '#60a5fa',
    'accent':   '#7c6cfa',
    'muted':    '#8888aa',
}

print('✅ Libraries loaded successfully')

## 1. Dataset — Real-world Review Corpus

In [ ]:
CORPUS = [
    ("Absolutely incredible product! Exceeded every expectation. Highly recommend.", "positive"),
    ("Great quality and fast delivery. Very satisfied with this purchase.", "positive"),
    ("The customer service was outstanding. They resolved my issue immediately.", "positive"),
    ("Best investment I've made. Works perfectly and looks amazing.", "positive"),
    ("Love this product! Simple, elegant, and incredibly reliable.", "positive"),
    ("Wonderful experience from start to finish. Will definitely buy again.", "positive"),
    ("Five stars! Brilliant design, easy setup, and excellent performance.", "positive"),
    ("Impressed by the quality. Far better than competing products.", "positive"),
    ("Terrible product. Broke after one week. Complete waste of money.", "negative"),
    ("Awful customer support. Nobody responded to my complaints for two weeks.", "negative"),
    ("Do not buy this. The description is misleading and quality is horrible.", "negative"),
    ("Worst purchase ever. Frustrating experience, nothing works as advertised.", "negative"),
    ("Very disappointed. Arrived damaged and the return process is a nightmare.", "negative"),
    ("Broken on arrival. Customer service was rude and unhelpful. Avoid.", "negative"),
    ("Poor build quality. Feels cheap and fragile. Not worth the price.", "negative"),
    ("Constant crashes and errors. The app is completely unusable.", "negative"),
    ("It's okay. Nothing special but does the job.", "neutral"),
    ("Average product. Delivery was on time. Price is reasonable.", "neutral"),
    ("Neither good nor bad. Works as described, nothing more.", "neutral"),
    ("Decent quality for the price. Some minor issues but acceptable.", "neutral"),
    ("Standard product. Expected more but not entirely disappointed.", "neutral"),
    ("Meets basic requirements. Not impressed but not complaining either.", "neutral"),
]

df = pd.DataFrame(CORPUS, columns=['text', 'label'])
df['word_count'] = df['text'].str.split().str.len()
df['char_count'] = df['text'].str.len()

print(f'Dataset: {len(df)} samples')
print(df['label'].value_counts().to_string())
df.head()

## 2. NLP Pipeline — Lexicon-based Analyzer

In [ ]:
POSITIVE_WORDS = {
    'great','good','excellent','amazing','wonderful','fantastic','love','best',
    'happy','joy','beautiful','awesome','perfect','brilliant','outstanding','superb',
    'delightful','pleased','impressed','enjoy','recommend','satisfied','helpful',
    'friendly','reliable','fast','easy','clean','smooth','efficient','innovative',
    'thrilled','excited','glad','grateful','thankful','positive','success','win',
    'incredible','five','stars','elegant','simple','impressive',
}

NEGATIVE_WORDS = {
    'bad','terrible','awful','horrible','poor','worst','hate','disappointed',
    'frustrating','annoying','useless','broken','slow','difficult','confusing',
    'expensive','waste','problem','issue','error','fail','failed','never','avoid',
    'boring','ugly','disgusting','pathetic','mediocre','inferior','inadequate',
    'disaster','regret','unhappy','angry','upset','annoyed','crashes','damaged',
    'misleading','rude','cheap','fragile','unusable','constant','errors',
}

NEGATION = {'not','no','never','neither','nor','barely','hardly','scarcely'}
INTENSIFIERS = {'very','really','extremely','absolutely','incredibly','so','quite','super','far'}

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

def analyze(text):
    tokens = tokenize(text)
    pos, neg, negated = 0.0, 0.0, False
    for i, t in enumerate(tokens):
        if t in NEGATION:
            negated = True; continue
        if i > 0 and tokens[i-1] not in NEGATION:
            negated = False
        mult = 1.5 if i > 0 and tokens[i-1] in INTENSIFIERS else 1.0
        if t in POSITIVE_WORDS:
            (neg if negated else pos).__add__  # just mark
            if negated: neg += mult
            else: pos += mult
        elif t in NEGATIVE_WORDS:
            if negated: pos += mult * 0.5
            else: neg += mult
    total = pos + neg
    compound = max(-1, min(1, (pos - neg) / (total + 1))) if total else 0.0
    label = 'positive' if compound >= 0.1 else 'negative' if compound <= -0.1 else 'neutral'
    score = 0.5 + compound * 0.5 if label == 'positive' else 0.5 - abs(compound) * 0.5 if label == 'negative' else 0.5
    confidence = min(0.99, abs(compound) + 0.3 * math.log(1 + total))
    return {'label': label, 'score': round(score, 4), 'compound': round(compound, 4),
            'confidence': round(confidence, 4), 'pos_hits': pos, 'neg_hits': neg}

df['prediction'] = df['text'].apply(lambda t: analyze(t)['label'])
df['score'] = df['text'].apply(lambda t: analyze(t)['score'])
df['confidence'] = df['text'].apply(lambda t: analyze(t)['confidence'])
df['compound'] = df['text'].apply(lambda t: analyze(t)['compound'])
df['correct'] = df['label'] == df['prediction']

accuracy = df['correct'].mean()
print(f'\n🎯 Model Accuracy: {accuracy:.1%}')
print(f'\nPer-class breakdown:')
print(df.groupby('label')['correct'].mean().apply(lambda x: f'{x:.1%}').to_string())

## 3. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('SentimentAI — Model Analytics Dashboard', fontsize=16, color='#f0f0f5', fontweight='bold', y=1.01)
fig.patch.set_facecolor('#0a0a0f')

# 1. Label distribution
ax = axes[0, 0]
counts = df['label'].value_counts()
bars = ax.bar(counts.index, counts.values,
              color=[COLORS[l] for l in counts.index],
              width=0.5, edgecolor='none', alpha=0.85)
ax.set_title('Label Distribution', color='#f0f0f5', pad=10)
ax.set_facecolor('#111118')
ax.spines[:].set_visible(False)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1,
            str(int(b.get_height())), ha='center', va='bottom', color='#f0f0f5', fontsize=12)

# 2. Compound score histogram
ax = axes[0, 1]
for label, grp in df.groupby('label'):
    ax.hist(grp['compound'], bins=8, color=COLORS[label], alpha=0.7, label=label, edgecolor='none')
ax.set_title('Compound Score Distribution', color='#f0f0f5', pad=10)
ax.set_facecolor('#111118')
ax.spines[:].set_visible(False)
ax.legend(framealpha=0, labelcolor='#f0f0f5', fontsize=9)
ax.axvline(0, color='#333344', linewidth=1, linestyle='--')

# 3. Confidence by class
ax = axes[0, 2]
for i, label in enumerate(['positive', 'negative', 'neutral']):
    vals = df[df['label'] == label]['confidence']
    bp = ax.boxplot(vals, positions=[i], patch_artist=True, widths=0.4,
                    boxprops=dict(facecolor=COLORS[label], alpha=0.6),
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(color=COLORS[label]),
                    capprops=dict(color=COLORS[label]),
                    flierprops=dict(marker='o', color=COLORS[label], alpha=0.4))
ax.set_xticks([0,1,2])
ax.set_xticklabels(['positive','negative','neutral'])
ax.set_title('Confidence by Class', color='#f0f0f5', pad=10)
ax.set_facecolor('#111118')
ax.spines[:].set_visible(False)
ax.set_ylim(0, 1.05)

# 4. Confusion matrix
ax = axes[1, 0]
labels_order = ['positive', 'negative', 'neutral']
cm = np.zeros((3, 3), dtype=int)
label_idx = {l: i for i, l in enumerate(labels_order)}
for _, row in df.iterrows():
    cm[label_idx[row['label']]][label_idx[row['prediction']]] += 1
im = ax.imshow(cm, cmap='Blues', aspect='auto', vmin=0)
ax.set_xticks([0,1,2]); ax.set_xticklabels(labels_order, fontsize=9)
ax.set_yticks([0,1,2]); ax.set_yticklabels(labels_order, fontsize=9)
ax.set_title('Confusion Matrix', color='#f0f0f5', pad=10)
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else '#8888aa', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', color='#8888aa'); ax.set_ylabel('Actual', color='#8888aa')

# 5. Score vs Confidence scatter
ax = axes[1, 1]
for label in ['positive', 'negative', 'neutral']:
    sub = df[df['label'] == label]
    ax.scatter(sub['score'], sub['confidence'], c=COLORS[label],
               label=label, alpha=0.8, s=80, edgecolors='none')
    # Wrong predictions
    wrong = sub[sub['label'] != sub['prediction']]
    if len(wrong):
        ax.scatter(wrong['score'], wrong['confidence'], c='white',
                   marker='x', s=120, linewidths=2, zorder=5)
ax.set_title('Score vs Confidence (✕ = misclassified)', color='#f0f0f5', pad=10)
ax.set_facecolor('#111118'); ax.spines[:].set_visible(False)
ax.legend(framealpha=0, labelcolor='#f0f0f5', fontsize=9)
ax.set_xlabel('Sentiment Score', color='#8888aa')
ax.set_ylabel('Confidence', color='#8888aa')

# 6. Word frequency heatmap
ax = axes[1, 2]
top_words = ['great','excellent','amazing','love','recommend',
             'terrible','awful','horrible','waste','broken',
             'okay','decent','average','standard','meets']
heat = np.zeros((3, len(top_words)))
for i, label in enumerate(['positive','negative','neutral']):
    texts = ' '.join(df[df['label']==label]['text'])
    tokens = tokenize(texts)
    counts = Counter(tokens)
    for j, w in enumerate(top_words):
        heat[i, j] = counts.get(w, 0)
im2 = ax.imshow(heat, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(top_words)))
ax.set_xticklabels(top_words, rotation=45, ha='right', fontsize=7)
ax.set_yticks([0,1,2]); ax.set_yticklabels(['pos','neg','neu'])
ax.set_title('Word Frequency by Class', color='#f0f0f5', pad=10)

plt.tight_layout()
plt.savefig('sentiment_analytics.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0f', edgecolor='none')
plt.show()
print('\n✅ Figure saved as sentiment_analytics.png')

## 4. Feature Engineering

In [ ]:
def extract_features(text):
    tokens = tokenize(text)
    pos_hits = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg_hits = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    intensifier_hits = sum(1 for t in tokens if t in INTENSIFIERS)
    negation_hits = sum(1 for t in tokens if t in NEGATION)
    excl = text.count('!')
    question = text.count('?')
    caps_ratio = sum(1 for c in text if c.isupper()) / max(len(text), 1)
    avg_word_len = np.mean([len(t) for t in tokens]) if tokens else 0
    result = analyze(text)
    return {
        'pos_hits': pos_hits,
        'neg_hits': neg_hits,
        'intensifier_hits': intensifier_hits,
        'negation_hits': negation_hits,
        'exclamations': excl,
        'questions': question,
        'caps_ratio': round(caps_ratio, 4),
        'avg_word_len': round(avg_word_len, 2),
        'word_count': len(tokens),
        'compound': result['compound'],
        'confidence': result['confidence'],
    }

features_df = pd.DataFrame([extract_features(t) for t in df['text']])
features_df['label'] = df['label'].values

print('Feature matrix shape:', features_df.shape)
features_df.groupby('label').mean().round(3)

## 5. Feature Importance (Manual SHAP-style)

In [ ]:
feature_cols = ['pos_hits','neg_hits','intensifier_hits','negation_hits',
                'exclamations','caps_ratio','avg_word_len','word_count']

# Compute correlation with compound score as a simple importance proxy
importances = {}
for col in feature_cols:
    corr = np.corrcoef(features_df[col], features_df['compound'])[0, 1]
    importances[col] = corr

imp_sorted = dict(sorted(importances.items(), key=lambda x: abs(x[1]), reverse=True))

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0a0a0f')
ax.set_facecolor('#111118')

names = list(imp_sorted.keys())
values = list(imp_sorted.values())
colors = [COLORS['positive'] if v > 0 else COLORS['negative'] for v in values]

bars = ax.barh(names, values, color=colors, alpha=0.85, edgecolor='none', height=0.6)
ax.axvline(0, color='#333344', linewidth=1)
ax.set_title('Feature Importance (Correlation with Sentiment Score)', color='#f0f0f5', fontsize=13, pad=12)
ax.spines[:].set_visible(False)
ax.set_xlabel('Pearson Correlation', color='#8888aa')

for bar, val in zip(bars, values):
    ax.text(val + (0.01 if val >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{val:+.2f}', va='center', ha='left' if val >= 0 else 'right',
            color='#f0f0f5', fontsize=9)

pos_p = mpatches.Patch(color=COLORS['positive'], alpha=0.85, label='Positive correlation')
neg_p = mpatches.Patch(color=COLORS['negative'], alpha=0.85, label='Negative correlation')
ax.legend(handles=[pos_p, neg_p], framealpha=0, labelcolor='#f0f0f5', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('✅ Feature importance chart saved')

## 6. Live API Demo

In [ ]:
import urllib.request, json as _json

test_texts = [
    "This is hands down the best product I have ever used!",
    "Terrible quality, do not waste your money on this.",
    "It works fine. Nothing to complain about but nothing to celebrate either.",
    "Not bad at all! Surprisingly good for the price.",
]

print('🔬 Running analysis...\n')
print(f'{"Text":<55} {"Label":<10} {"Score":<7} {"Conf"}')
print('─' * 85)

for text in test_texts:
    try:
        payload = _json.dumps({'text': text}).encode()
        req = urllib.request.Request(
            'http://localhost:8000/analyze',
            data=payload, method='POST',
            headers={'Content-Type': 'application/json'}
        )
        with urllib.request.urlopen(req, timeout=2) as r:
            result = _json.loads(r.read())['sentiment']
        source = 'API'
    except Exception:
        result = analyze(text)
        source = 'local'

    label = result['label']
    emoji = '✅' if label == 'positive' else '❌' if label == 'negative' else '➖'
    print(f"{text[:52]:<55} {emoji} {label:<8} {result['score']:.3f}  {result['confidence']:.3f}  [{source}]")

print('\n✅ Analysis complete')

## 7. Trend Simulation

In [ ]:
random.seed(42)
days = 30
dates = [datetime.now() - timedelta(days=days-i) for i in range(days)]

pos_trend, neg_trend, vol_trend = [], [], []
p = 0.55
for _ in range(days):
    p = max(0.2, min(0.85, p + random.gauss(0, 0.04)))
    pos_trend.append(p)
    neg_trend.append((1 - p) * random.uniform(0.5, 0.8))
    vol_trend.append(random.randint(50, 300))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]})
fig.patch.set_facecolor('#0a0a0f')

for ax in [ax1, ax2]:
    ax.set_facecolor('#111118')
    ax.spines[:].set_visible(False)

ax1.fill_between(dates, pos_trend, alpha=0.15, color=COLORS['positive'])
ax1.plot(dates, pos_trend, color=COLORS['positive'], linewidth=2.5, label='Positive')
ax1.fill_between(dates, neg_trend, alpha=0.15, color=COLORS['negative'])
ax1.plot(dates, neg_trend, color=COLORS['negative'], linewidth=2, linestyle='--', label='Negative')
ax1.set_title('30-Day Sentiment Trend', color='#f0f0f5', fontsize=13, pad=12)
ax1.set_ylabel('Sentiment Score', color='#8888aa')
ax1.legend(framealpha=0, labelcolor='#f0f0f5')
ax1.set_ylim(0, 1)
ax1.grid(axis='y', alpha=0.3)

ax2.bar(dates, vol_trend, color=COLORS['accent'], alpha=0.6, width=0.8, edgecolor='none')
ax2.set_ylabel('Volume', color='#8888aa')
ax2.set_xlabel('Date', color='#8888aa')

plt.tight_layout()
plt.savefig('trend_chart.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()
print('✅ Trend chart saved')